# 04 — Analyses statistiques : questions Q1 à Q6

**Projet NutriHFr** · Étapes 4 et 5 de la consigne (méthodes, résultats, interprétation)

| Question | Indicateur | Méthode | Visualisation |
|---|---|---|---|
| Q1 Énergie | apport énergétique (kcal/j) par individu | statistiques descriptives pondérées, IC 95 % | histogramme + densité |
| Q2 Sexe | apports par sexe | différence de moyennes pondérées (test z) | boxplots |
| Q3 Âge | apports par classe d'âge | moyennes pondérées + régression pondérée | courbes par classe |
| Q4 Densité | apport pour 1 000 kcal, % de l'énergie | rapport nutriment / énergie | barres |
| Q5 Contributeurs | % de contribution des groupes | **ratio des moyennes** pondéré | barres empilées |
| Q6 Variabilité | moyenne, médiane, P5–P95 ; part intra/inter | percentiles pondérés, ANOVA | violin plot |

**Conventions** : population = adultes 18-79 ans ; apport d'un individu = moyenne de ses journées ; toutes les estimations sont **pondérées** (`poids_enquete`) ; IC 95 % par linéarisation (sans prise en compte des strates du plan de sondage, d'où des IC légèrement trop étroits — limite à citer).

In [ ]:
# --- Configuration commune à tous les notebooks ------------------------------
import sys
import warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=RuntimeWarning)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nutrihfr import config as C
from nutrihfr.viz import set_style, savefig, PALETTE, SEX_COLORS
set_style()
if C.MODE == "synthetic" and not C.raw_dir("inca3").exists():   # 1re utilisation : données de démo
    import runpy; runpy.run_path(str(ROOT / "scripts" / "make_synthetic_data.py"), run_name="__main__")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 50)
print("Mode des données :", C.MODE, "->", C.raw_dir("inca3"))

In [ ]:
from nutrihfr.pipeline import NutriPipeline
from nutrihfr.stats import describe_weighted, weighted_mean_diff_test, wls, weighted_quantile, variance_components
from nutrihfr.intake import IntakeCalculator
from nutrihfr.viz import top_n_with_other
from nutrihfr.lineage import Lineage

res = NutriPipeline(verbose=False).run(save=False)
df, daily, lines, NUTS = res["indiv"], res["daily"], res["lines"], res["nutrients"]
LAB = {k: f"{v[0]} ({v[1]}/j)" for k, v in C.NUTRIENTS.items()}
C.TABLES.mkdir(parents=True, exist_ok=True)
print(f"{len(df)} adultes analysés (n Hommes = {(df.sexe == 'Homme').sum()}, n Femmes = {(df.sexe == 'Femme').sum()})")

## Q1 — Distribution de l'apport énergétique journalier

In [ ]:
q1 = pd.concat([describe_weighted(df, "energie_kcal").assign(sexe="Ensemble"),
                describe_weighted(df, "energie_kcal", by="sexe")], ignore_index=True)
q1 = q1.set_index("sexe")
q1.to_csv(C.TABLES / "Q1_energie.csv")
q1

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for s in ["Homme", "Femme"]:
    d = df[df.sexe == s]
    ax.hist(d["energie_kcal"], bins=40, weights=d["poids_enquete"], density=True,
            alpha=0.35, color=SEX_COLORS[s], label=f"{s} (n={len(d)})")
    sns.kdeplot(x=d["energie_kcal"], weights=d["poids_enquete"], ax=ax, color=SEX_COLORS[s], lw=2,
                bw_adjust=1.0, common_norm=False, label=None, clip=(0, None))
    ax.axvline(q1.loc[s, "mediane"], color=SEX_COLORS[s], ls="--", lw=1)
ax.set(title="Q1 · Apport énergétique journalier (pondéré)", xlabel="kcal / jour", ylabel="densité")
ax.legend(); savefig(fig, "Q1_energie_distribution"); plt.show()

In [ ]:
e = q1.loc["Ensemble"]
print(f"L'apport énergétique moyen des adultes est de {e.moyenne:.0f} kcal/j "
      f"(IC 95 % : {e.ic95_bas:.0f}-{e.ic95_haut:.0f}), médiane {e.mediane:.0f} kcal/j ; "
      f"90 % des adultes se situent entre {e.P5:.0f} et {e.P95:.0f} kcal/j.")
skew = df["energie_kcal"].skew()
print(f"Asymétrie = {skew:.2f} -> distribution {'étalée vers la droite' if skew > 0.3 else 'à peu près symétrique'}.")

**À retenir (Q1)** — *à réécrire avec tes chiffres réels.* La moyenne et la médiane sont proches si la distribution est symétrique ; une longue queue à gauche (apports très bas) est un premier signal de **sous-déclaration** (analysée dans le notebook 05). Ces valeurs sont des apports **observés sur 3 jours** et non des apports habituels.

## Q2 — Différences entre hommes et femmes

In [ ]:
rows = []
for n in NUTS:
    t = describe_weighted(df, n, by="sexe").set_index("sexe")
    test = weighted_mean_diff_test(df, n, "sexe")          # Femme - Homme (ordre alphabétique)
    rows.append({"nutriment": LAB[n], "moy_Femmes": t.loc["Femme", "moyenne"], "moy_Hommes": t.loc["Homme", "moyenne"],
                 "diff_F-H": test["difference"], "ic95_bas": test["ic95_bas"], "ic95_haut": test["ic95_haut"],
                 "ecart_relatif_%": 100 * test["difference"] / t.loc["Homme", "moyenne"], "p_value": test["p_value"]})
q2 = pd.DataFrame(rows).set_index("nutriment")
q2.to_csv(C.TABLES / "Q2_sexe.csv")
q2.style.format("{:.2f}").format("{:.1e}", subset=["p_value"])

In [ ]:
show = ["energie_kcal", "proteines_g", "fibres_g", "calcium_mg", "fer_mg", "vitc_mg"]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, n in zip(axes.flat, show):
    sns.boxplot(data=df, x="classe_age", y=n, hue="sexe", palette=SEX_COLORS, ax=ax, showfliers=False,
                width=0.6, linewidth=1, order=C.AGE_LABELS)
    ax.set(title=LAB[n], xlabel="", ylabel="")
    ax.legend_.remove() if ax is not axes.flat[0] else ax.legend(title="")
fig.suptitle("Q2-Q3 · Apports par sexe et classe d'âge (boîtes non pondérées, sans valeurs extrêmes)", y=1.01)
plt.tight_layout(); savefig(fig, "Q2Q3_boxplots"); plt.show()

**À retenir (Q2)** — Les hommes ont des apports **absolus** plus élevés pour presque tous les nutriments, surtout parce qu'ils mangent plus (énergie). La vraie question est donc : *à énergie égale*, les femmes ont-elles une alimentation différente ? → Q4.

*Note de méthode* : avec ~1 000 personnes par groupe, presque toutes les différences sont « significatives » (p < 0,05) ; on commente surtout la **taille de l'écart** (colonne `ecart_relatif_%`) et son IC.

## Q3 — Évolution avec l'âge

In [ ]:
q3 = pd.concat({n: describe_weighted(df, n, by=["sexe", "classe_age"])[["sexe", "classe_age", "n", "moyenne", "ic95_bas", "ic95_haut"]]
                for n in NUTS}, names=["nutriment"]).reset_index(level=0)
q3.to_csv(C.TABLES / "Q3_age.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, n in zip(axes, ["energie_kcal", "calcium_mg", "fibres_g"]):
    d = q3[q3.nutriment == n]
    for s in ["Homme", "Femme"]:
        ds = d[d.sexe == s].set_index("classe_age").loc[C.AGE_LABELS]
        ax.errorbar(ds.index, ds["moyenne"], yerr=[ds["moyenne"] - ds["ic95_bas"], ds["ic95_haut"] - ds["moyenne"]],
                    marker="o", ms=6, lw=2, capsize=3, color=SEX_COLORS[s], label=s)
    ax.set(title=LAB[n], xlabel="classe d'âge")
axes[0].legend(); fig.suptitle("Q3 · Moyennes pondérées et IC 95 % par classe d'âge", y=1.03)
plt.tight_layout(); savefig(fig, "Q3_age"); plt.show()

**Régression linéaire pondérée** : `apport ~ sexe + classe d'âge`. Elle quantifie l'effet de l'âge **à sexe égal** (et inversement). Référence : femmes de 18-24 ans. Erreurs-types robustes (HC1).

In [ ]:
reg = {}
for n in ["energie_kcal", "calcium_mg", "fibres_g", "vitc_mg"]:
    r = wls(df, n, ["sexe", "classe_age"])
    reg[LAB[n]] = r["coef"].round(1).astype(str) + "  [" + r["ic95_bas"].round(1).astype(str) + " ; " + r["ic95_haut"].round(1).astype(str) + "]"
    print(f"{n:<14} R² pondéré = {r.attrs['r2_pondere']:.3f}  (n = {r.attrs['n']})")
q3_reg = pd.DataFrame(reg)
q3_reg.to_csv(C.TABLES / "Q3_regression.csv")
q3_reg

**À retenir (Q3)** — Lire chaque coefficient comme « *différence moyenne par rapport à une femme de 18-24 ans, toutes choses égales par ailleurs* ». Le R² faible est normal : l'âge et le sexe n'expliquent qu'une petite partie des différences d'apport entre individus (cf. Q6).

## Q4 — Densité nutritionnelle (à énergie comparable)

In [ ]:
dens_cols = [c for c in df.columns if c.endswith("_per1000kcal")]
q4 = pd.concat({c.replace("_per1000kcal", ""): describe_weighted(df, c, by="sexe").set_index("sexe")["moyenne"]
                for c in dens_cols}, axis=1).T
q4["ecart_F_vs_H_%"] = 100 * (q4["Femme"] / q4["Homme"] - 1)
q4.to_csv(C.TABLES / "Q4_densite.csv")
q4

In [ ]:
d = q4.drop(index=[i for i in ["alcool_g"] if i in q4.index])
fig, ax = plt.subplots(figsize=(8, 4))
ax.axvline(0, color="#52514e", lw=1)
colors = [SEX_COLORS["Femme"] if v > 0 else SEX_COLORS["Homme"] for v in d["ecart_F_vs_H_%"]]
ax.barh([C.NUTRIENTS[i][0] for i in d.index], d["ecart_F_vs_H_%"], color=colors, height=0.6)
ax.set(title="Q4 · Densité nutritionnelle : écart Femmes vs Hommes (pour 1 000 kcal)",
       xlabel="% d'écart (> 0 : plus dense chez les femmes)")
savefig(fig, "Q4_densite"); plt.show()

pct = describe_weighted(df.melt(id_vars=["sexe", "poids_enquete"], value_vars=["pct_ae_proteines", "pct_ae_glucides", "pct_ae_lipides"]),
                        "value", by=["sexe", "variable"])
pct.pivot(index="variable", columns="sexe", values="moyenne").assign(
    reference_ANSES=["40-55 %", "35-40 %", "10-20 %"])

**À retenir (Q4)** — Rapporter à 1 000 kcal neutralise le fait que les hommes mangent plus. Si un nutriment reste plus élevé chez les femmes pour 1 000 kcal, c'est que leur alimentation est **qualitativement** plus riche en ce nutriment (par ex. plus de fruits et légumes → vitamine C, fibres). La répartition de l'énergie entre macronutriments se compare aux références de l'Anses (2016) indiquées dans la dernière colonne.

## Q5 — Principaux aliments contributeurs

**Méthode : ratio des moyennes (pondéré).** Contribution du groupe *g* = $\sum_i w_i \cdot apport_{i,g} \;/\; \sum_i w_i \cdot apport_i$.

*Pourquoi pas la moyenne des ratios ?* La moyenne des ratios calcule d'abord la part du groupe **pour chaque individu**, puis en fait la moyenne : chaque personne pèse autant, qu'elle consomme 200 ou 2 000 mg de calcium, et les parts deviennent instables chez les petits consommateurs. Le ratio des moyennes répond à la question de santé publique « *d'où vient le calcium consommé par la population ?* » et ses parts somment exactement à 100 %. On compare les deux à la fin.

In [ ]:
contrib = {}
for n in C.CONTRIB_NUTRIENTS:
    c = IntakeCalculator.contributions(lines, df, n)
    contrib[n] = c.set_index("groupe")["contribution_pct"]
contrib = pd.DataFrame(contrib).fillna(0)
contrib.to_csv(C.TABLES / "Q5_contributeurs.csv")
contrib.round(1).sort_values("proteines_g", ascending=False).head(12)

In [ ]:
# Couleurs fixes par groupe : les 7 groupes les plus contributeurs en moyenne, le reste en "Autres"
main = contrib.mean(axis=1).sort_values(ascending=False).index[:7].tolist()
colors = {g: PALETTE[k] for k, g in enumerate(main)}
stack = contrib.loc[main].T
stack["Autres"] = 100 - stack.sum(axis=1)
fig, ax = plt.subplots(figsize=(10, 4.2))
left = np.zeros(len(stack))
ylabels = [C.NUTRIENTS[n][0] for n in stack.index]
for g in stack.columns:
    v = stack[g].to_numpy()
    ax.barh(ylabels, v, left=left, color=colors.get(g, "#b5b4ad"), edgecolor="white", linewidth=2,
            height=0.65, label=g)
    for y, (l, x) in enumerate(zip(left, v)):
        if x >= 8:
            ax.text(l + x / 2, y, f"{x:.0f} %", ha="center", va="center", fontsize=8, color="white")
    left += v
ax.invert_yaxis(); ax.grid(False); ax.set_xlim(0, 100)
ax.set(title="Q5 · Contribution des groupes d'aliments (ratio des moyennes, pondéré)",
       xlabel="% de l'apport total de la population")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.18), fontsize=8.5)
plt.tight_layout(); savefig(fig, "Q5_contributeurs"); plt.show()

In [ ]:
# Comparaison des deux méthodes pour les protéines
rm = IntakeCalculator.contributions(lines, df, "proteines_g").set_index("groupe")["contribution_pct"]
mr = IntakeCalculator.contributions_mean_of_ratios(lines, df, "proteines_g").set_index("groupe")["contribution_pct"]
comp = pd.DataFrame({"ratio_des_moyennes": rm, "moyenne_des_ratios": mr}).fillna(0)
comp["difference_points"] = comp["moyenne_des_ratios"] - comp["ratio_des_moyennes"]
comp.sort_values("ratio_des_moyennes", ascending=False).head(10).round(1)

**À retenir (Q5)** — Pour chaque nutriment, nommer les 2-3 premiers groupes et leur part cumulée (« *les produits laitiers apportent X % du calcium* »). Les écarts entre les deux méthodes sont souvent faibles, mais ils montrent l'importance de **préciser la méthode** quand on publie une contribution.

## Q6 — Variabilité interindividuelle (exemple : femmes de 25-44 ans)

Deux sources de variabilité :
- **inter-individuelle** : des personnes différentes mangent différemment (c'est ce qui nous intéresse) ;
- **intra-individuelle** : la même personne mange différemment d'un jour à l'autre.

Avec seulement 3 jours, la distribution des moyennes individuelles contient encore une part de variabilité intra, ce qui **élargit artificiellement** la distribution (P5 trop bas, P95 trop haut). On mesure cette part par une décomposition de variance (ANOVA à un facteur).

In [ ]:
grp = df[(df.sexe == "Femme") & (df.classe_age == "25-44")]
q6 = pd.concat({LAB[n]: describe_weighted(grp, n).iloc[0] for n in NUTS}, axis=1).T
q6["CV_%"] = 100 * q6["ecart_type"] / q6["moyenne"]
q6.to_csv(C.TABLES / "Q6_variabilite_F25_44.csv")
print(f"Femmes 25-44 ans : n = {len(grp)}")
q6[["n", "moyenne", "mediane", "P5", "P25", "P75", "P95", "CV_%"]]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, n in zip(axes, ["energie_kcal", "calcium_mg", "vitc_mg"]):
    sns.violinplot(data=df, x="classe_age", y=n, hue="sexe", split=True, palette=SEX_COLORS, ax=ax,
                   inner="quartile", cut=0, linewidth=1, order=C.AGE_LABELS, density_norm="width")
    ax.set(title=LAB[n], xlabel="", ylabel="")
    ax.legend_.remove() if ax is not axes[0] else ax.legend(title="")
fig.suptitle("Q6 · Dispersion des apports (violin plots, quartiles en pointillés)", y=1.02)
plt.tight_layout(); savefig(fig, "Q6_violin"); plt.show()

In [ ]:
vc = pd.DataFrame({LAB[n]: variance_components(daily[daily.id_indiv.isin(grp.id_indiv)], n) for n in NUTS}).T
vc[["part_inter_pct", "ratio_intra_inter", "n_individus", "n_jours_moyen"]]

**À retenir (Q6)** — Une part inter-individuelle de 30 % signifie que 70 % de la variance d'un jour d'apport vient des variations **d'un jour à l'autre chez la même personne**. Les nutriments très « épisodiques » (vitamine C, présente surtout dans certains fruits) ont un ratio intra/inter élevé. Pour estimer correctement la proportion de personnes sous un seuil de besoin, il faudrait des méthodes d'**apport habituel** (NCI, SPADE) — mentionné comme piste d'amélioration.

## Lignage d'un résultat (annexe C)

In [ ]:
m = describe_weighted(grp, "calcium_mg").iloc[0]
lin = Lineage(resultat=f"Calcium moyen = {m.moyenne:.0f} mg/j [IC95 {m.ic95_bas:.0f}-{m.ic95_haut:.0f}]",
              population={"Sexe": "Femme", "Âge": "25-44", "n": int(m.n)})
print(lin.tree())
(C.TABLES / "lineage_calcium_F25_44.json").write_text(lin.to_json(), encoding="utf-8")

➡️ Suite : `05_sensibilite_sous_declaration.ipynb`